(searching)=

# Searching

```{contents}
:local:
:depth: 2
```

```{index} Duplicates Turn One Position into an Interval
```



```{index} Trace the Boundary Rather than Stopping at Equality
```

## Trace the Boundary Rather than Stopping at Equality


Chapter 16's any-match binary search can establish that a target exists. A report requiring every duplicate needs two boundaries. With sorted values 2,4,4,4,7 and target four, lower bound is the first value at least four; upper bound is the first value greater than four. Their half-open interval includes all three fours.

Maintain a candidate boundary interval [low,high], represented by the loop's half-open search range [low,high). High begins at Length; a result equal to Length is legal even though it is not a valid element index. Each comparison at mid removes a region whose relation to the target is already known.

| Search | low | high | mid/value | Update |
| --- | ---: | ---: | --- | --- |
| Lower | 0 | 5 | 2 / 4 | high=2; an earlier equal may exist |
| Lower | 0 | 2 | 1 / 4 | high=1 |
| Lower | 0 | 1 | 0 / 2 | low=1; 2 is strictly below target |
| Upper | 0 | 5 | 2 / 4 | low=3; equality belongs before upper bound |
| Upper | 3 | 5 | 4 / 7 | high=4 |
| Upper | 3 | 4 | 3 / 4 | low=4 |

Both loops stop when low equals high. Equality moves high in the lower search, but moves low past mid in the upper search. The next cell makes that single policy difference explicit; using the same equality action for both endpoints would lose the duplicate range.


In [1]:
using System;
static int Boundary(int[] values, int target, bool upper)
{
    int low = 0, high = values.Length;
    while (low < high)
    {
        int mid = low + (high - low) / 2;
        bool beforeBoundary = values[mid] < target || (upper && values[mid] == target);
        if (beforeBoundary) low = mid + 1;
        else high = mid;
    }
    return low;
}
int[] values = { 2, 4, 4, 4, 7 };
foreach (int target in new[] { 4, 5, 9 })
{
    int first = Boundary(values, target, false);
    int after = Boundary(values, target, true);
    Console.WriteLine($"{target}: [{first},{after}), count {after - first}");
}


4: [1,4), count 3
5: [4,4), count 0
9: [5,5), count 0


For five and nine, the empty intervals represent no matches. Do not read values[first] until first is in range and the result contract requires that access. Merely finding a valid insertion boundary does not establish equality.

Guided task: trace both searches for an empty array and then for 4,4,4 with target four. Explain why an all-equal input is a strong test of the comparison at equality. Then attach invoice IDs to the equal amounts and enumerate every ID in [first,after), preserving the distinction between sorted positions and original arrival positions.

```{dropdown} Check the boundary cases
Both endpoints of an empty array are zero without any element read. For three fours, lower is zero and upper is three, so every position belongs to the match interval. Stopping at the first equal midpoint would not establish either required boundary.
```

The focused loop counts neither timing nor comparer calls. The retained implementations below add explicit policies, instrumentation and assignment variations; compare them under the same contract before making workload recommendations.


**Common mistake.** Returning `mid` as soon as `values[mid] == target`, as the any-match search does. For the lower bound that can land on the second or third 4; the boundary searches must keep moving `high` or `low` past equal values until the range is empty.


```{index} Duplicates Turn One Position into an Interval
```

## Duplicates Turn One Position into an Interval



A sorted array [2, 4, 4, 4, 7] contains three occurrences of four. An any-match search can return position one, two or three. An invoice report that needs every matching amount needs the whole interval, not a fortunate midpoint result.

Define lower bound as the first position whose value is at least the target, and upper bound as the first position whose value is greater than it. For target four, those positions are one and four, so the matching half-open interval is [1,4), with length three. For missing target five, both bounds are four; the empty interval represents no matches. A bound can equal Length, which is a boundary position rather than a valid element index.

| Target | Lower bound | Upper bound | Matching count |
| --- | --- | --- | --- |
| 1 | 0 | 0 | 0 |
| 4 | 1 | 4 | 3 |
| 5 | 4 | 4 | 0 |
| 9 | 5 | 5 | 0 |

The two algorithms differ at equality. Lower-bound search must keep the possibility of an earlier equal value; upper-bound search must move beyond all equal values. Copying one comparison into both versions can silently return a single endpoint twice. Trace an all-equal array and targets just below/above it to expose that mistake.

Ordering and equality policies must match the report's meaning. If invoices are sorted by amount, equal amounts are not necessarily the same invoice identity. If a search uses a different string comparer from sorting, the eliminated-region argument may fail. The returned interval identifies positions in this ordered representation, not original arrival positions unless that information is stored separately.

Guided task: manually find both bounds for target four in the example, then replace the three fours with labeled invoices. List the report IDs that should be returned and explain why a single any-match result is insufficient. Finally, account for preparing the sorted snapshot when comparing a batch of these queries with repeated scans.

For related explanation, see the further reading below.[^development-reading]


First-match linear search, iterative binary search, missing-target traces, and library return values are taught in {doc}`Chapter 16 <../16_searching_sorting/1601_searching>`. This optional extension studies equality policies, duplicate boundaries, and repeated-query workloads.


```{index} Equality and Ordering Policies
```

## Equality and Ordering Policies




```{index} Equality and Ordering Policies
```

Choose a supplier-name policy explicitly. Ordinal equality is case-sensitive; ordinal case-insensitive equality can treat names with different case as one key. Use an ordering compatible with that equivalence when sorting for binary search. String comparisons need not be constant-time in character length.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] names={"Acme","BETA","acme","Delta"};
foreach(var policy in new[]{StringComparer.Ordinal,StringComparer.OrdinalIgnoreCase})
{
 int index=ReportSearch.LinearFirst(names,"ACME",policy,out long probes);
 Console.WriteLine($"first={index}, probes={probes}");
}
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


Comparison policy is part of the contract, not an incidental culture setting. For records, search the intended key; equality of whole records may be a different business rule. Binary searches below count calls to a three-way comparer, not each primitive relational operation inside that comparer.

**Common mistake.** Sorting with `StringComparer.Ordinal` and then searching with `OrdinalIgnoreCase`. The sorted order no longer matches the search's idea of equal, so binary search can skip the block where `acme` and `Acme` sit; the comparer is part of the data's contract, not a per-call option.


```{index} Lower and Upper Bounds for Duplicates
```

## Lower and Upper Bounds for Duplicates




```{index} Lower and Upper Bounds for Duplicates
```

A lower bound is the first position with value ≥ target; an upper bound is the first position with value > target. Both can equal n. Lower-bound invariants: every index below low is < target, and every index at or above high is ≥ target. At low=high, that boundary is established. Upper bound replaces those relations with ≤ and >. Equality must keep searching rather than return early.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,200,200,300};
foreach(int target in new[]{50,200,250,500})
{
 int lower=ReportSearch.LowerBound(sorted,target,Comparer<int>.Default,out long lp);
 int upper=ReportSearch.UpperBound(sorted,target,Comparer<int>.Default,out long up);
 bool found=lower<sorted.Length && sorted[lower]==target;
 Console.WriteLine($"target={target}, lower={lower}, upper={upper}, count={upper-lower}, first={(found?lower:-1)}, probes={lp+up}");
}
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


Equal records occupy [lower,upper), so upper−lower counts occurrences and lower identifies the first sorted-order match. Two boundary searches cost Θ(log n) worst-case probes; enumerating k matching records adds Θ(k) output work. The original arrival position cannot be recovered from a sorted amount-only array: retain record IDs or original indexes.

**Common mistake.** Reporting `lower` as a found position without checking `sorted[lower] == target`. For the missing target 250 the lower bound is 4, a legal index that holds 300; the `found` test in the cell is what separates "first match" from "insertion point".


```{index} Repeated-Query Workloads
```

## Repeated-Query Workloads




```{index} searching; workload costs
```


For q lookups, scanning costs up to Θ(qn). Sorting once and searching costs Θ(n log n+q log n), assuming a comparison sort and static data. Updates can invalidate the sorted snapshot or require maintenance. Sorting merely to answer one query may add work; an existing sorted array can support binary search even for one query. Dictionary exact-key lookup needs its own duplicate and update policy; ordered ranges and all matches are different requirements.

```{index} Exercise
```

## Exercise




```{index} Exercise
```

Return the full sorted match range for an invoice amount, including no-match and empty cases. Verify lower/upper boundaries before indexing.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,200,300};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sorted={100,200,200,300};
foreach(int target in new[]{200,250})
{
 int lower=ReportSearch.LowerBound(sorted,target,Comparer<int>.Default,out _);
 int upper=ReportSearch.UpperBound(sorted,target,Comparer<int>.Default,out _);
 Console.WriteLine($"target={target}, range=[{lower},{upper}), count={upper-lower}");
}
Console.WriteLine($"empty lower={ReportSearch.LowerBound(Array.Empty<int>(),200,Comparer<int>.Default,out _)}");
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


target=200, range=[1,3), count=2
target=250, range=[3,3), count=0
empty lower=0


The output is `target=200, range=[1,3), count=2`, `target=250, range=[3,3), count=0`, and `empty lower=0`. For the missing 250 both bounds land on index 3, the position where it would be inserted, so the count is zero without any index being read; on an empty array both bounds are 0.


```{rubric} Footnotes
```


[^2301_searching-1]: [Microsoft’s Array.BinarySearch contract](https://learn.microsoft.com/en-us/dotnet/api/system.array.binarysearch?view=net-10.0) specifies negative insertion-position encoding and sorted-input requirements.
[^2301_searching-2]: Probe counts assume constant-cost access and comparison; variable-length keys need an additional key-length cost model.

[^development-reading]: Further reading: [OpenDSA, Search in Sorted Arrays](https://opendsa.cs.vt.edu/OpenDSA/Books/Catalog/html/SortedSearch.html). The worked business scenarios and guided reasoning here are original teaching examples.
